# <font color='green'>Fine Tunning Open Source LLM for Chatbot Entity List Assistant</font>

## Packages

In [ ]:
!pip install -q bitsandbytes datasets accelerate loralib evaluate

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git@main git+https://github.com/huggingface/peft.git

In [ ]:
!pip install -U torchao

## Imports

In [ ]:
import os
import json
import torch
import evaluate
import torch.nn as nn
import transformers
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, pipeline
from peft import LoraConfig, get_peft_model
from datasets import Dataset, Features, ClassLabel, Value, Sequence
import warnings
warnings.filterwarnings('ignore')

## Check GPU

In [ ]:
if torch.cuda.is_available():
  print('Nº of GPUs:', torch.cuda.device_count())
  print('GPU Name:', torch.cuda.get_device_name(0))
  print('GPU Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Quantization Parameters

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
    llm_int8_enable_fp32_cpu_offload = True
    )

# Load Tokenizer and Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-7b",
    quantization_config = quantization_config,
    device_map = 'auto'
  )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-7b")

## Freeze Original Weights

In [ ]:
# Loop
for param in model.parameters():
  param.requires_grad = False
  if param.ndim == 1:
    param.data = param.data.to(torch.float32)

## Gradient Checkpoint

In [ ]:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

## Tensor Conversion

In [ ]:
class CastOutputFloat(nn.Sequential):
  def forward(self, x):
    return super().forward(x).to(torch.float32)

model.lm_head = CastOutputFloat(model.lm_head)

## Fine Tuning Parameters

In [ ]:
# LoRa Config
config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

In [ ]:
model = get_peft_model(model, config)

In [ ]:
# Print training parameters
def print_trainable_parameters(model):
  trainable_params = 0
  all_param = 0
  for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
      trainable_params += param.numel()
  print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")

In [ ]:
print_trainable_parameters(model)

## Load Data

In [ ]:
file_1 = open("/content/drive/Othercomputers/Meu laptop/Agentic_AI/LLM_PLN/cap_10/dataset_1.json")
data_1 = json.load(file_1)

In [ ]:
data_1

In [ ]:
# List for question and answer
questions = []
answers = []

In [ ]:
for i in data_1["questions"]:
  questions += [i["question"]]
  answers += [i["answer"]]

In [ ]:
# Original Dataset
data_1["questions"][0]

In [ ]:
# First Question
questions[0]

In [ ]:
# Format data for model training
dataset = Dataset.from_dict({
    "id": list(range(len(questions))),
    "questions": questions, # Changed from 'question' to 'questions'
    "answers": answers      # Changed from 'answer' to 'answers'
    },
    features = Features({
      "id": Value(dtype = 'string'),
      "questions": Value(dtype = 'string'),
      "answers": Value(dtype = 'string')
    })
)

In [ ]:
# Divide data into train and test
dataset = dataset.train_test_split(test_size = 0.15)

In [ ]:
# Merge questions and answers
def merge_columns(register):
  register["output"] = register["questions"] + " ->: " + register["answers"]
  return register

In [ ]:
train_dataset = dataset.map(merge_columns)

In [ ]:
# Show Format
train_dataset["train"]["output"][0]

In [ ]:
train_dataset["train"][0]

In [ ]:
# Tokenizing data
train_dataset = train_dataset.map(lambda samples: tokenizer(samples['output']), batched = True)

In [ ]:
# Tokenized data
train_dataset["train"][0]

## Setting training args

In [ ]:
if tokenizer.pad_token == None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model_trainer = transformers.Trainer(
    model = model,
    train_dataset = train_dataset["train"],
    eval_dataset = train_dataset["test"],
    args = transformers.TrainingArguments(
        eval_strategy = "epoch",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = 'outputs',
        report_to = "none"
    ),
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm = False)
)

## Training the model

In [ ]:
model.config.use_cache = False
model_trainer.train()

## Evaluating Model Performance

In [ ]:
def model_predict(question):
  model.eval()
  device = next(model.parameters()).device
  batch = tokenizer(
      f"{question} ->: ",
      return_tensors = 'pt',
      padding = True,
      truncation = True
      )
  batch = {k: v.to(device) for k, v in batch.items()}
  with torch.no_grad(), torch.cuda.amp.autocast():
    output_tokens = model.generate(
        **batch,
        max_new_tokens = 50,
        pad_token_id = tokenizer.eos_token_id
    )
  return tokenizer.decode(output_tokens[0], skip_special_tokens = True)

In [ ]:
# List of predictions
predictions = []

In [ ]:
for i in train_dataset["test"]["questions"]:
  predictions.append(model_predict(i))

In [ ]:
# Bleu method
bleu = evaluate.load('bleu')

In [ ]:
#Extract real data
real_data = train_dataset["test"]["output"]

In [ ]:
results = bleu.compute(predictions = predictions, references = real_data)

In [ ]:
results

## Deploy of the model

In [ ]:
# Device
device = next(model.parameters()).device

In [ ]:
question_1 = 'How can I see if a company is in the Entity List?'

In [ ]:
tokenized_question = tokenizer(question_1, return_tensors = "pt", padding = True, truncation = True)

In [ ]:
tokenized_question = {name: tensor.to(device) for name, tensor in tokenized_question.items()}

In [ ]:
tokenized_question

In [ ]:
# Generate answer
with torch.no_grad(), torch.cuda.amp.autocast():
  predict_tokens = model.generate(**tokenized_question, max_new_tokens = 100, pad_token_id = tokenizer.eos_token_id)

In [ ]:
# Decode anwser
tokenizer.decode(predict_tokens[0], skip_special_tokens = True)

In [ ]:
# Saving Model
torch.save(model.state_dict(), "/content/drive/MyDrive/Modelos_LLMs/Fine_Tuning_Falcon_Entity_List/trained_model.pt")